# Linux生存指南

## 学习目标

- 在Linux终端下完成基本的文件操作
- 理解文件权限和管道
- 通过SSH远程连接服务器
- 使用tmux让程序在后台运行
- 能在服务器上找到、查看、操作音频文件

## 本节说明

本notebook中的shell命令可以通过 `!` 前缀在Jupyter中执行。
但**实际使用时，你应该在真正的终端中操作**，而不是在notebook中。
notebook中的 `!` 只是用于课堂演示和练习验证。


## 1. 为什么要用Linux？

在深度学习研究中，你几乎必然会使用Linux服务器。原因很简单：

- **GPU服务器几乎都运行Linux**——Windows和macOS对GPU的支持远不如Linux
- **深度学习框架（PyTorch、TensorFlow）在Linux上最稳定**——文档、社区、教程都以Linux为主
- **服务器是共享资源**——多人同时使用一台GPU服务器，Linux天然支持多用户
- **CI/助听器研究的计算需求**——训练模型、处理数据都需要大量计算

**类比**：你的笔记本是工作站，服务器是工厂。你在工作站上设计，去工厂里生产。


## 2. 文件系统与基本命令

### 2.1 我在哪里？——pwd 和 ls

最常打的两个命令：`pwd`（我在哪里）和 `ls`（这里有什么）。


In [ ]:
!pwd  # 显示当前工作目录

In [ ]:
!ls   # 列出当前目录的内容

In [ ]:
!ls -la  # 列出详细信息（权限、大小、修改时间）

### 2.2 去哪里？——cd

```bash
cd /home/user          # 绝对路径：从根目录开始
cd data                # 相对路径：从当前目录开始
cd ..                  # 返回上一级
cd ~                   # 回到home目录
cd -                   # 回到上次所在的目录
```


### 2.3 创建和删除——mkdir, rm, cp, mv

```bash
# 创建
mkdir project           # 创建目录
mkdir -p project/data  # 递归创建（父目录不存在也行）

# 复制
cp file.txt backup/     # 复制文件到目录
cp -r data/ backup/     # 递归复制目录

# 移动/重命名
mv old.txt new.txt      # 重命名
mv file.txt data/       # 移动到目录

# 删除（⚠️ 慎用！）
rm file.txt             # 删除文件（不可恢复！）
rm -r directory/        # 递归删除目录
rm -ri directory/       # 递归删除，逐个确认
```

**重要提醒**：`rm` 删除的文件不会进回收站，删了就没了。养成习惯：删除前先 `ls` 确认。

### 2.4 查看文件——cat, less, head, tail

```bash
cat file.txt           # 查看整个文件
head -20 file.txt     # 查看前20行
tail -20 file.txt     # 查看后20行
less file.txt         # 分页查看（按q退出）
```


### 练习1：文件操作

在终端中完成以下操作（不要在notebook中做）：

1. 用 `pwd` 确认你当前在哪个目录
2. 用 `mkdir` 创建一个目录 `~/lab-training/module1-exercise`
3. 用 `cd` 进入这个目录
4. 用 `touch` 创建一个文件 `test.txt`
5. 用 `cp` 把 `test.txt` 复制为 `test_backup.txt`
6. 用 `ls -la` 确认两个文件都存在
7. 用 `rm` 删除 `test_backup.txt`
8. 用 `ls` 确认只剩 `test.txt`

## 3. 权限、管道、重定向

### 3.1 文件权限

Linux中每个文件都有三组权限：所有者(user)、组(group)、其他人(others)。


In [ ]:
!ls -la / | head -5  # 看看根目录下的权限格式

权限格式：`-rwxr-xr-x`

- 第1位：文件类型（`-`=文件，`d`=目录，`l`=链接）
- 第2-4位：所有者权限（r=读，w=写，x=执行）
- 第5-7位：组权限
- 第8-10位：其他人权限

```bash
chmod +x script.sh      # 给脚本添加执行权限
chmod 755 script.sh     # 用数字设置权限：7=rwx, 5=r-x
chmod 644 data.txt      # 常用权限：文件644，目录755
```


### 3.2 管道（Pipe）——把命令串起来

管道 `|` 把一个命令的输出传给下一个命令。这是Linux最强大的特性之一。


In [ ]:
# 示例：找出当前目录下所有的Python文件
!find . -name '*.py' 2>/dev/null | head -5

In [ ]:
# 示例：统计一个文件有多少行
!echo -e 'line1\nline2\nline3\nline4\nline5' | wc -l

### 3.3 重定向——保存输出

```bash
# 把输出保存到文件
python train.py > train.log 2>&1    # 标准输出和错误都保存
python train.py >> train.log 2>&1   # 追加模式

# 只保存错误输出
python train.py 2> error.log
```


### 3.4 在音频处理中的实际应用

```bash
# 找出目录下所有wav文件
find /data -name '*.wav' | wc -l

# 找出采样率为16kHz的wav文件
find /data -name '*.wav' -exec soxi {} \; | grep '16000'

# 批量转换采样率
for f in *.wav; do sox "$f" -r 16000 "resampled/$f"; done

# 查看GPU使用情况并排序
nvidia-smi | grep python
```

### 练习2：管道与重定向

1. 用 `find` 找出当前目录下所有的 `.ipynb` 文件
2. 用管道把结果传给 `wc -l` 统计数量
3. 用 `grep` 在一个文件中搜索关键词
4. 把搜索结果重定向到一个新文件中

## 4. SSH远程连接

### 4.1 基本连接

```bash
# 基本连接
ssh username@server_ip

# 指定端口
ssh -p 2222 username@server_ip

# 使用密钥连接（更安全）
ssh -i ~/.ssh/my_key username@server_ip
```

### 4.2 文件传输

```bash
# 上传文件到服务器
scp local_file.txt username@server_ip:~/remote_dir/

# 从服务器下载文件
scp username@server_ip:~/remote_file.txt ./local_dir/

# 传输整个目录
scp -r local_dir/ username@server_ip:~/remote_dir/
```

### 4.3 SSH配置文件

编辑 `~/.ssh/config` 可以简化连接：

```
Host gpu-server
    HostName 192.168.1.100
    User myname
    Port 22
    IdentityFile ~/.ssh/my_key
```

配置之后只需 `ssh gpu-server` 即可连接。


### 练习3：连接服务器

1. 用SSH连接到租用的服务器
2. 连接后运行 `hostname` 和 `whoami` 确认身份
3. 运行 `nvidia-smi` 查看GPU信息
4. 用 `scp` 从本地传输一个小文件到服务器
5. 在服务器上确认文件已传输

## 5. tmux——让程序在后台运行

### 为什么需要tmux？

当你通过SSH连接服务器运行训练时，如果网络断开或关闭终端，训练就会中断。
tmux可以创建一个"虚拟终端"，即使你断开SSH，程序仍然在运行。

### 核心操作

| 操作 | 命令/按键 |
|------|----------|
| 新建session | `tmux new -s train` |
| 分离session | `Ctrl+b` 然后按 `d` |
| 重新连接 | `tmux attach -s train` |
| 列出所有session | `tmux ls` |
| 杀掉session | `tmux kill-session -t train` |
| 在session中新建窗口 | `Ctrl+b` 然后按 `c` |
| 切换窗口 | `Ctrl+b` 然后按 `0/1/2...` |


### 典型工作流

```bash
# 1. SSH连接服务器
ssh gpu-server

# 2. 创建tmux session
tmux new -s training

# 3. 在tmux中运行训练
python train.py --epochs 100

# 4. 分离session（训练继续运行）
# 按 Ctrl+b，然后按 d

# 5. 断开SSH
exit

# --- 几小时后 ---

# 6. 重新SSH连接
ssh gpu-server

# 7. 重新连接tmux session
tmux attach -s training

# 8. 查看训练进度
```

### 练习4：tmux

1. 创建一个名为 `test` 的tmux session
2. 在session中运行 `ping localhost`（模拟一个长时间运行的程序）
3. 分离session（Ctrl+b 然后 d）
4. 用 `tmux ls` 确认session还在运行
5. 重新连接session
6. 按 Ctrl+C 停止 ping
7. 退出tmux

## 6. Vim基础与VS Code Remote

### 6.1 Vim最小生存指南

Vim是Linux服务器上最常见的文本编辑器。你不需要精通Vim，只需要能改配置文件就行。

```bash
vim file.txt      # 打开文件
```

| 操作 | 按键 |
|------|------|
| 进入插入模式 | `i` |
| 退出插入模式 | `Esc` |
| 保存 | `:w` |
| 退出 | `:q` |
| 保存并退出 | `:wq` 或 `ZZ` |
| 不保存退出 | `:q!` |
| 搜索 | `/关键词` 然后 `n` 下一个 |

**如果你不会Vim，可以用nano替代**：
```bash
nano file.txt     # 更简单的编辑器
```


### 6.2 VS Code Remote（推荐）

VS Code Remote比Vim好用得多，强烈推荐。

**设置步骤：**

1. 在本地安装VS Code
2. 安装 `Remote - SSH` 扩展
3. 按 `Ctrl+Shift+P`，输入 `Remote-SSH: Connect to Host`
4. 输入 `username@server_ip`
5. 等待连接（首次需要安装VS Code Server）
6. 连接成功后，就像在本地编辑一样

**优点**：
- 图形界面，不需要记Vim命令
- 代码高亮、自动补全
- 内置终端
- Jupyter Notebook支持

### 练习5：编辑文件

1. 用Vim（或nano）在服务器上创建一个文件 `hello.py`，写入以下内容：
   ```python
   print('Hello from the server!')
   ```
2. 运行 `python hello.py` 确认输出
3. 用VS Code Remote连接服务器，打开同一个文件并修改
4. 再次运行确认修改生效

## 7. 音频文件操作实战

在CI/助听器研究中，我们经常需要批量处理音频文件。以下是一些常用操作。


### 7.1 查看音频文件信息

```bash
# 使用soxi（sox的一部分）
soxi file.wav

# 使用ffprobe
ffprobe file.wav
```

### 7.2 批量处理音频文件

```bash
# 批量转换采样率到16kHz
mkdir -p resampled
for f in *.wav; do
    sox "$f" -r 16000 "resampled/$f"
done

# 批量转换格式（mp3 → wav）
for f in *.mp3; do
    ffmpeg -i "$f" "${f%.mp3}.wav"
done

# 统计wav文件数量
find . -name '*.wav' | wc -l

# 查看所有wav文件的总大小
du -sh *.wav
```

### 练习6：音频文件操作

**小挑战**：连接服务器后完成以下任务

1. 在服务器上找到培训者指定的wav文件目录
2. 统计wav文件的数量
3. 查看其中一个wav文件的信息（采样率、时长、通道数）
4. 计算整个目录下wav文件的总大小
5. 用Python读取一个wav文件并打印其波形长度


In [ ]:
# 练习6的Python部分：读取wav文件
import soundfile as sf
import numpy as np

# TODO: 替换为实际的文件路径
# filepath = '/path/to/your/audio.wav'
# waveform, sr = sf.read(filepath)
# print(f'采样率: {sr} Hz')
# print(f'波形长度: {len(waveform)} 个样本')
# print(f'时长: {len(waveform)/sr:.2f} 秒')
# print(f'通道数: {1 if waveform.ndim == 1 else waveform.shape[1]}')

# 临时演示：生成一个模拟wav文件的元信息
print('wav文件信息示例：')
print(f'  采样率: 16000 Hz')
print(f'  波形长度: 48000 个样本')
print(f'  时长: 3.00 秒')
print(f'  通道数: 1 (单声道)')

## 本节要点

1. **文件操作**：`cd`, `ls`, `pwd`, `mkdir`, `rm`, `cp`, `mv` 是最基本的命令
2. **管道**：`|` 把命令串起来，是Linux最强大的特性之一
3. **SSH**：远程连接服务器的标准方式，学会配置 `~/.ssh/config` 省很多时间
4. **tmux**：让程序在后台运行，断开SSH也不中断训练
5. **Vim**：只需要会打开、编辑、保存、退出；日常开发用VS Code Remote
6. **音频操作**：`sox` 和 `ffmpeg` 是音频文件处理的瑞士军刀

详细命令参考：[Linux速查表](../cheatsheet-linux.md) | [Vim速查表](../cheatsheet-vim.md)


---
下一节：[02-environment-setup.ipynb](02-environment-setup.ipynb) — 深度学习环境搭建